# Local HF Server + Qwen 3.5 0.8B × Gymnasium

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/simple-jev/blob/main/notebooks/Ollama_Jev_Gymnasium.ipynb)

**Quick start:** Select **Runtime → Change runtime type → T4 GPU** (or another available GPU), then **Runtime → Run all**. CPU also works, more slowly. Use a current Python 3.12+ runtime.

This notebook installs Simple Jev's **Hugging Face server**, downloads **[Qwen/Qwen3.5-0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B)**, and starts the server on `127.0.0.1` **inside the same Colab runtime**. No server on your laptop, Ollama installation, tunnel, or hosted inference API is needed. Internet access is needed for installation and the initial model download; inference runs in the notebook's runtime.

The default experiment compares Qwen via **`/v1/classifier`** with random actions and a hand-written CartPole controller, then displays rewards and videos. Simple Jev scores allowed next-token labels and builds the response; it does not generate a JSON completion. The legacy notebook filename is retained so existing links keep working.

HalfCheetah running is optional: enable its checkbox near the end. A general language model is not a trained locomotion policy. Simulated time pauses during API requests, so smooth video playback is not evidence of real-time control. Results are exploratory, with no pre-filled model scores.

In [ ]:
#@title 1. Install the HF server and simulation dependencies
import os, sys, json, pathlib, subprocess, shutil

if sys.version_info < (3, 12):
    raise RuntimeError("Simple Jev requires Python 3.12+. Select a current Colab runtime and reconnect.")

REPO_URL = "https://github.com/vtavakkoli/simple-jev.git"
# Pin the server code inspected for this notebook; model weights use their current HF revision.
SERVER_REVISION = "92f0ba913f37f1946084531b7ed3d6d4f0ab81e4"
WORK_DIR = pathlib.Path("/content" if pathlib.Path("/content").is_dir() else pathlib.Path.cwd())
REPO_DIR = WORK_DIR / ("simple-jev-colab-" + SERVER_REVISION[:12])
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", SERVER_REVISION], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", SERVER_REVISION], check=True)
actual_revision = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if actual_revision != SERVER_REVISION:
    raise RuntimeError(f"Unexpected checkout at {REPO_DIR}. Use a fresh runtime or a different REPO_DIR.")

# pip retains an already-compatible Colab CUDA PyTorch installation.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR / "hf-server"),
                "gymnasium[classic-control,mujoco]>=1.0,<2", "requests", "numpy",
                "matplotlib", "imageio", "imageio-ffmpeg"], check=True)

# Offscreen rendering works on Colab CPU and GPU, without an X display.
if sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
    if shutil.which("apt-get"):
        privilege = [] if os.geteuid() == 0 else ["sudo"]
        subprocess.run(privilege + ["apt-get", "update", "-qq"], check=True)
        subprocess.run(privilege + ["apt-get", "install", "-y", "-qq", "libosmesa6", "libgl1", "libegl1"], check=True)
    os.environ.setdefault("MUJOCO_GL", "osmesa")
    os.environ.setdefault("PYOPENGL_PLATFORM", os.environ["MUJOCO_GL"])
    os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
print("Installed server revision:", actual_revision)

## 2. Configuration
The default model is `Qwen/Qwen3.5-0.8B`. The next cell starts it locally and assigns `JEV_URL` automatically. The first startup downloads the weights and may take several minutes.

One seed and 100 steps give a short first demonstration. For a larger comparison, set `SEEDS = [0, 1, 2]` and increase `MAX_STEPS`. Normal episode limits are CartPole 500, Pendulum 200, and HalfCheetah 1000. Use held-out seeds for final evaluation. GPU inference uses BF16 when supported; otherwise FP32 is selected for conservative numerical behavior, including on T4 and CPU.

When changing `ENV_ID`, the default baselines adapt to the task. Disable `RECORD_VIDEO` if your local machine cannot render offscreen. Rendering variables are configured before importing Gymnasium/MuJoCo; reconnect if you previously imported MuJoCo with another rendering backend.

In [ ]:
#@title 2. Choose the experiment
JEV_MODEL = "Qwen/Qwen3.5-0.8B" #@param {type:"string"}
ENV_ID = "CartPole-v1" #@param ["CartPole-v1", "Pendulum-v1", "HalfCheetah-v5"]
METHODS = ["random", "heuristic", "jev"] if ENV_ID == "CartPole-v1" else ["random", "zero", "jev"]
SEEDS = [0]
MAX_STEPS = 100 #@param {type:"integer"}
ACTION_REPEAT = 1
RECORD_VIDEO = True #@param {type:"boolean"}
HTTP_TIMEOUT = 180
SERVER_STARTUP_TIMEOUT = 900

In [ ]:
#@title 3. Start the local HF server and wait for the model
import socket, time, requests

def stop_hf_server():
    """Stop only the child process started by this notebook."""
    process = globals().get("hf_server_process")
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)

def wait_for_hf_server(process, url, model, log_path, timeout=900):
    deadline = time.monotonic() + timeout
    next_update = 0
    while time.monotonic() < deadline:
        if process.poll() is not None:
            tail = log_path.read_text(errors="replace")[-8000:]
            raise RuntimeError(f"HF server exited with code {process.returncode}.\n{tail}")
        try:
            response = requests.get(url + "/health", timeout=2)
            if response.ok:
                health = response.json()
                if health.get("status") == "ready" and health.get("model") == model:
                    return health
                raise RuntimeError(f"Unexpected server on {url}: {health}")
        except requests.RequestException:
            pass
        if time.monotonic() >= next_update:
            print("Downloading/loading the HF model; server log:", log_path, flush=True)
            next_update = time.monotonic() + 15
        time.sleep(1)
    raise TimeoutError(f"HF server did not become ready in {timeout}s.\n" + log_path.read_text(errors="replace")[-8000:])

# Detect hardware in a short-lived child; the notebook does not load model weights.
hardware_probe = """import json, torch
gpu = torch.cuda.is_available()
bf16 = gpu and torch.cuda.is_bf16_supported()
print(json.dumps({'device': 'cuda' if gpu else 'cpu', 'dtype': 'bfloat16' if bf16 else 'float32',
                  'hardware': torch.cuda.get_device_name(0) if gpu else 'CPU', 'torch': torch.__version__}))
"""
hardware = json.loads(subprocess.check_output([sys.executable, "-c", hardware_probe], text=True))
print("HF server hardware:", hardware)
stop_hf_server()
with socket.socket() as free_socket:
    free_socket.bind(("127.0.0.1", 0))
    JEV_PORT = free_socket.getsockname()[1]
JEV_URL = f"http://127.0.0.1:{JEV_PORT}"
HF_SERVER_LOG = WORK_DIR / "jev_hf_server.log"
server_command = [sys.executable, "-u", str(REPO_DIR / "hf-server" / "hf_server.py"),
                  "--model", JEV_MODEL, "--device", hardware["device"], "--dtype", hardware["dtype"],
                  "--host", "127.0.0.1", "--port", str(JEV_PORT), "--max-model-len", "4096",
                  "--max-batch-size", "4", "--max-batch-tokens", "4096"]
server_env = os.environ.copy()
server_env["PYTHONUNBUFFERED"] = "1"
with HF_SERVER_LOG.open("w") as log_handle:
    hf_server_process = subprocess.Popen(server_command, cwd=REPO_DIR, env=server_env,
                                         stdout=log_handle, stderr=subprocess.STDOUT)
try:
    health = wait_for_hf_server(hf_server_process, JEV_URL, JEV_MODEL, HF_SERVER_LOG, SERVER_STARTUP_TIMEOUT)
except BaseException:
    stop_hf_server()
    raise
print("Ready:", health, "at", JEV_URL)

In [ ]:
#@title 4. Define the environments and benchmark
import collections, csv, datetime, json, pathlib, platform, time, uuid
import gymnasium as gym
import numpy as np
import requests
import imageio.v2 as imageio

TASKS = {
    "CartPole-v1": {
        "names": ["cart_position_m", "cart_velocity_mps", "pole_angle_rad", "pole_angular_velocity_radps"],
        "goal": "Keep the pole upright and the cart near the center. Angle zero is upright; positive angle leans right. Positive cart position is right. Avoid pole falling or cart leaving track.",
        "joints": ["push"], "levels": {"left": 0, "right": 1},
    },
    "Pendulum-v1": {
        "names": ["cos_angle", "sin_angle", "angular_velocity_radps"],
        "goal": "Swing the pendulum upright and stabilize it at angle zero (cos=1, sin=0), using little torque. Positive torque increases angular velocity.",
        "joints": ["torque"], "levels": {"n2": -2., "n1": -1., "zero": 0., "p1": 1., "p2": 2.},
    },
    "HalfCheetah-v5": {
        "names": ["root_z", "root_pitch", "back_thigh_angle", "back_shin_angle", "back_foot_angle", "front_thigh_angle", "front_shin_angle", "front_foot_angle", "root_x_velocity", "root_z_velocity", "root_pitch_velocity", "back_thigh_velocity", "back_shin_velocity", "back_foot_velocity", "front_thigh_velocity", "front_shin_velocity", "front_foot_velocity"],
        "goal": "Coordinate the six joints to run forward in positive x. Reward is forward velocity minus 0.1 times the sum of squared torques. Build a repeated gait from observation and recent history. No trained gait or future-state simulation is supplied.",
        "joints": ["back_thigh", "back_shin", "back_foot", "front_thigh", "front_shin", "front_foot"],
        "levels": {"n1": -1., "n05": -.5, "zero": 0., "p05": .5, "p1": 1.},
    },
}

class DecisionClient:
    def __init__(self, backend, model, url):
        self.backend, self.model, self.url = backend, model, url.rstrip("/")
        self.http = requests.Session()
        self.server_metadata = {}
        if backend != "jev":
            raise ValueError("This notebook uses the local Simple Jev HF server")
        r = self.http.get(self.url + "/health", timeout=HTTP_TIMEOUT)
        r.raise_for_status()
        health = r.json()
        if health.get("status") != "ready" or health.get("model") != self.model:
            raise RuntimeError(f"Unexpected model/server: {health}")
        self.server_metadata = {"health": health, "server_revision": SERVER_REVISION,
                                "runtime": hardware}
        print(f"{backend}: {self.model} at {self.url}")

    def decide(self, env_id, observation, history, step):
        task = TASKS[env_id]
        obs = np.asarray(observation)
        if obs.shape != (len(task["names"]),) or not np.isfinite(obs).all():
            raise ValueError("Unexpected or nonfinite observation; default environment configuration required.")
        state = {"environment": env_id, "step": step,
                 "observation": dict(zip(task["names"], np.round(obs, 5).tolist())),
                 "recent_history": list(history)}
        t0 = time.perf_counter()
        questions = {j: {"type": "choice", "instructions": task["goal"] + " Choose the control for " + j,
                          "criteria": {k: "Apply " + str(v) for k, v in task["levels"].items()}} for j in task["joints"]}
        body = {"model": self.model, "state": state, "questions": questions}
        r = self.http.post(self.url + "/v1/classifier", json=body, timeout=HTTP_TIMEOUT)
        if not r.ok:
            raise RuntimeError(f"HF classifier HTTP {r.status_code}: {r.text[:2000]}")
        raw = r.json()
        choices = {j: raw["answers"][j]["choice"] for j in task["joints"]}
        usage = raw.get("usage", {})
        if not isinstance(choices, dict) or set(choices) != set(task["joints"]):
            raise ValueError("Missing or extra action fields")
        if any(not isinstance(v, str) or v not in task["levels"] for v in choices.values()):
            raise ValueError("Invalid action label")
        values = [task["levels"][choices[j]] for j in task["joints"]]
        action = int(values[0]) if env_id == "CartPole-v1" else np.asarray(values, dtype=np.float32)
        return action, {"latency_ms": (time.perf_counter()-t0)*1000, "choices": choices, "usage": usage, "raw": raw}

def baseline_action(method, env_id, obs, rng):
    task = TASKS[env_id]
    if method == "heuristic":
        if env_id != "CartPole-v1":
            raise ValueError("The heuristic baseline supports only CartPole")
        # Hand-written feedback controller, explicitly separate from model actions.
        x, dx, theta, dtheta = obs
        return int(theta + .25*dtheta + .015*x + .025*dx > 0)
    if method == "zero":
        if env_id == "CartPole-v1":
            raise ValueError("CartPole has no zero-force action")
        return np.zeros(len(task["joints"]), dtype=np.float32)
    if method != "random":
        raise ValueError("Unknown method " + method)
    values = list(task["levels"].values())
    action = rng.choice(values, size=len(task["joints"]))
    return int(action[0]) if env_id == "CartPole-v1" else action.astype(np.float32)

def run_benchmark(env_id, methods, seeds, max_steps, action_repeat=1, record_video=True):
    if env_id not in TASKS or max_steps < 1 or action_repeat < 1 or not seeds or not methods:
        raise ValueError("Check environment, methods, seeds and positive limits")
    if len(set(methods)) != len(methods) or set(methods) - {"jev", "random", "zero", "heuristic"}:
        raise ValueError("Unknown or repeated methods")
    if "heuristic" in methods and env_id != "CartPole-v1":
        raise ValueError("Use random/zero baselines for this environment")
    if "zero" in methods and env_id == "CartPole-v1":
        raise ValueError("CartPole has no zero-force action")
    out = pathlib.Path("jev_gym_results") / (datetime.datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
    out.mkdir(parents=True)
    config = {"env_id": env_id, "methods": methods, "seeds": list(seeds), "max_steps": max_steps,
              "action_repeat": action_repeat, "gymnasium": gym.__version__, "numpy": np.__version__,
              "python": platform.python_version(), "task": TASKS[env_id], "mode": "synchronous_simulation",
              "history_length": 4, "clients": {}}
    clients = {}
    for method in methods:
        if method == "jev":
            client = DecisionClient(method, JEV_MODEL, JEV_URL)
            clients[method] = client
            config["clients"][method] = {"model": client.model, "url": client.url, **client.server_metadata}
    (out/"config.json").write_text(json.dumps(config, indent=2))
    # Warm-up is logged separately and excluded from episode latency/reward.
    for method, client in clients.items():
        env = gym.make(env_id)
        try:
            obs, _ = env.reset(seed=int(seeds[0]))
            _, meta = client.decide(env_id, obs, [], 0)
            (out/(method+"_warmup.json")).write_text(json.dumps(meta, indent=2))
        finally:
            env.close()
    rows = []
    for method in methods:
        for episode, seed in enumerate(seeds):
            video = record_video and episode == 0
            env = gym.make(env_id, render_mode="rgb_array" if video else None)
            writer = None
            history = collections.deque(maxlen=4)
            rng = np.random.default_rng(int(seed)+10000)
            latencies, speeds = [], []
            reward_sum = 0.; steps = calls = failures = 0
            terminated = truncated = False
            status, error = "ok", ""
            started = time.perf_counter()
            trace_path = out/f"{method}_seed{seed}.jsonl"
            try:
                obs, info = env.reset(seed=int(seed))
                if video:
                    writer = imageio.get_writer(str(out/f"{method}_seed{seed}.mp4"), fps=env.metadata.get("render_fps", 30), macro_block_size=1)
                    writer.append_data(env.render())
                with trace_path.open("w") as trace:
                    while steps < max_steps and not (terminated or truncated):
                        before = np.asarray(obs).tolist()
                        try:
                            if method in clients:
                                calls += 1
                                action, meta = clients[method].decide(env_id, obs, history, steps)
                                latencies.append(meta["latency_ms"])
                            else:
                                action = baseline_action(method, env_id, obs, rng)
                                meta = {}
                            if not env.action_space.contains(action):
                                raise ValueError("Action outside environment action space")
                        except Exception as exc:
                            failures += 1
                            raise RuntimeError("Decision failed; no fallback action was applied: " + str(exc)) from exc
                        action_json = np.asarray(action).tolist()
                        step_rewards = []
                        for _ in range(min(action_repeat, max_steps-steps)):
                            obs, reward, terminated, truncated, info = env.step(action)
                            steps += 1
                            reward_sum += float(reward)
                            step_rewards.append(float(reward))
                            if env_id == "HalfCheetah-v5":
                                speeds.append(float(obs[8]))
                            if writer is not None:
                                writer.append_data(env.render())
                            if terminated or truncated:
                                break
                        history.append({"observation": np.round(before, 5).tolist(), "action": action_json, "reward": sum(step_rewards)})
                        trace.write(json.dumps({"step": steps, "before": before, "action": action_json, "after": np.asarray(obs).tolist(), "step_rewards": step_rewards, "terminated": bool(terminated), "truncated": bool(truncated), **meta})+"\n")
                        trace.flush()
                        if steps % 25 == 0:
                            print(f"{env_id} {method} seed={seed}: {steps} steps, reward={reward_sum:.2f}", flush=True)
            except Exception as exc:
                status, error = "failed", repr(exc)
                print(error)
            finally:
                env.close()
                if writer is not None:
                    writer.close()
            row = {"method": method, "seed": int(seed), "status": status, "return": reward_sum, "steps": steps,
                   "terminated": bool(terminated), "truncated": bool(truncated),
                   "capped_by_benchmark": steps >= max_steps and not (terminated or truncated),
                   "api_calls": calls, "decision_failures": failures,
                   "latency_mean_ms": float(np.mean(latencies)) if latencies else None,
                   "latency_p95_ms": float(np.percentile(latencies, 95)) if latencies else None,
                   "mean_forward_speed_mps": float(np.mean(speeds)) if speeds else None,
                   "wall_seconds": time.perf_counter()-started, "error": error}
            rows.append(row)
            with (out/"episodes.csv").open("w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=list(row)); w.writeheader(); w.writerows(rows)
            print(row)
            if status == "failed":
                print("Stopping this method after failure; partial episode is excluded from success summaries.")
                break
    for client in clients.values():
        client.http.close()
    print("Results:", out.resolve())
    return rows, out

In [ ]:
#@title 5. Inspect one real model decision before benchmarking
probe_client = DecisionClient("jev", JEV_MODEL, JEV_URL)
probe_env = gym.make(ENV_ID)
try:
    observation, _ = probe_env.reset(seed=int(SEEDS[0]))
    action, details = probe_client.decide(ENV_ID, observation, [], 0)
    if not probe_env.action_space.contains(action):
        raise ValueError("The server returned an invalid environment action")
    print("Model:", JEV_MODEL)
    print("Action:", np.asarray(action).tolist())
    print("First-call latency (ms):", round(details["latency_ms"], 2))
    print(json.dumps(details["raw"], indent=2))
finally:
    probe_env.close()
    probe_client.http.close()

## 6. Run and compare
The next cells run Qwen through the local HF server, compare it with the selected baselines, and display the first episode's video for each method. Random actions use the same discrete levels as the model. Failed episodes are labeled and excluded from successful-return summaries; no baseline silently replaces a failed model decision.

Warm-up requests are logged separately. `jev_gym_results/` contains model/server configuration, CSV rewards and latency, raw decision traces, and MP4 videos. The sample decision above and the warm-up are outside measured episodes. If the sample request fails, inspect `jev_hf_server.log` before proceeding.

In [ ]:
rows, output_dir = run_benchmark(ENV_ID, METHODS, SEEDS, MAX_STEPS, ACTION_REPEAT, RECORD_VIDEO)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Video
successful = [r for r in rows if r["status"] == "ok"]
labels = list(dict.fromkeys(r["method"] for r in successful))
if labels:
    means = [np.mean([r["return"] for r in successful if r["method"] == m]) for m in labels]
    stds = [np.std([r["return"] for r in successful if r["method"] == m]) for m in labels]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, means, yerr=stds, capsize=5)
    ax.set(ylabel="Episode return (mean ± population SD)", title="Exploratory results; successful episodes only")
    plt.show()
for m in METHODS:
    good = sum(r["status"] == "ok" for r in rows if r["method"] == m)
    failed = sum(r["status"] != "ok" for r in rows if r["method"] == m)
    print(m, "successful:", good, "failed:", failed, "planned:", len(SEEDS))
for path in sorted(output_dir.glob("*.mp4")):
    print(path.name)
    display(Video(str(path), embed=True, width=640))

## 7. Optional running experiment: HalfCheetah
Enable `RUN_HALFCHEETAH` below and execute the cell to compare Qwen with random and zero-torque baselines. Six joint questions are sent together in one native Jev request, with each torque quantized to five levels. No trained gait is supplied.

The first run uses one seed and 100 steps. Increase to 1000 steps for full episodes, and use multiple held-out seeds when comparing policies. HalfCheetah requires coordinated control; the 0.8B model may perform poorly. The displayed video runs at simulation speed, independent of the time spent waiting for inference.

In [ ]:
#@title 7. Try HalfCheetah running
RUN_HALFCHEETAH = False #@param {type:"boolean"}
if RUN_HALFCHEETAH:
    running_rows, running_dir = run_benchmark(
        "HalfCheetah-v5", ["random", "zero", "jev"],
        seeds=SEEDS, max_steps=MAX_STEPS, action_repeat=ACTION_REPEAT, record_video=RECORD_VIDEO,
    )
    for path in sorted(running_dir.glob("*.mp4")):
        print(path.name)
        display(Video(str(path), embed=True, width=640))
else:
    print("Enable RUN_HALFCHEETAH and rerun this cell to test running.")

## Interpretation and troubleshooting
- Compare identical seeds, horizons, observations, action levels, and repeat counts. The default short run demonstrates the pipeline, not statistically reliable control quality.
- Native Jev's label distributions are not calibrated probabilities of correctness. Reward and failure rate must be measured independently of confidence.
- Reward, forward speed, latency, and failure counts measure different things. Compare against a trained PPO/SAC controller before making locomotion claims; one is not included here.
- The local server loads the HF model only once. Re-running cell 3 stops the notebook's previous server before starting another, avoiding duplicate model allocations. A disconnected/deleted Colab runtime must run setup again.
- For startup failures, run `print(HF_SERVER_LOG.read_text()[-8000:])`. The startup cell also shows the log tail on exit or timeout. It will not silently switch models or use a remote endpoint.
- If you run out of GPU memory, stop other model processes or reconnect to a fresh runtime. FP32 on T4/CPU uses more memory than BF16 on supported GPUs.
- To stop inference and release its model memory when finished, run `stop_hf_server()` in a new cell.

References: [Simple Jev HF server](https://github.com/vtavakkoli/simple-jev/tree/main/hf-server), [Qwen 3.5 0.8B](https://huggingface.co/Qwen/Qwen3.5-0.8B), [HalfCheetah](https://gymnasium.farama.org/environments/mujoco/half_cheetah/).